# Experiment 9 - Part 1: Tools and the Agent Loop

**Course:** HAAI302COM - Artificial Intelligence Laboratory

A language model alone can only produce text. An **agent** is a language model that can also *act*: it decides to call a function (a **tool**), reads the result, and continues until it can answer. In this notebook we build three tools (a calculator, a Python code runner and a web search), look at exactly what the model sends when it wants to use one, write the agent loop by hand in a dozen lines, and measure how the quality of a tool's *description* changes whether the model picks the right tool.

The next notebook turns the hand-written loop into a **LangGraph** graph with memory; the third adds human approval checkpoints.

## 0. Setup

`agent_tools.py` holds the three tools and the Gemini model (OpenAI-compatible endpoint, used through LangChain's `ChatOpenAI`). Model answers are cached in `llm_cache.json` and search results in `search_cache.json`, so the notebook re-runs without internet or an API key.

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import pandas as pd
import agent_tools as at
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 170)
at.enable_cache()                       # cache model responses on disk
llm = at.get_llm()                      # gemini-2.5-flash-lite, temperature 0
print([t.name for t in at.TOOLS])

['calculator', 'run_python', 'web_search']


## 1. A tool is a function plus a description

The `@tool` decorator turns a Python function into something the model can call. LangChain reads three things from it: the **name**, the **docstring** (the model's only information about when to use it) and the **argument schema** (derived from the type hints). This schema is sent to the model as JSON on every request. We print it for the calculator.

In [2]:
for t in at.TOOLS:
    print(f"{t.name:12} args={list(t.args)}  description: {t.description[:75]}...")
print()
print(json.dumps(at.calculator.args_schema.model_json_schema()["properties"], indent=1))

calculator   args=['expression']  description: Evaluate an arithmetic expression exactly, for example '48271 * 9371' or 's...
run_python   args=['code']  description: Run Python code and return what it prints. Use it for date arithmetic, stri...
web_search   args=['query']  description: Search the web and return the top results (title, snippet, link). Use it fo...

{
 "expression": {
  "title": "Expression",
  "type": "string"
 }
}


**What to notice:** the model sees only three things about each tool: its name, the docstring and a small JSON schema (here one string argument called 'expression'). The schema comes from the Python type hints and the description from the docstring, so the docstring is effectively the tool's manual. The calculator docstring gives examples and lists the supported operations for a reason: the model cannot read the code.

## 2. Testing each tool alone, including how it fails

Before an LLM touches a tool, we test it by hand, with good input and with bad input. A tool in an agent should never crash the loop: it should return an **error message as text**, so the model can read it and try again.

- **calculator**: an arithmetic parser built on Python's `ast` module. It refuses anything that is not a number or an allowed operation.
- **run_python**: starts a separate Python process with a 5 second time limit.

In [3]:
tests = [("calculator", "48271 * 9371"), ("calculator", "sqrt(2)"), ("calculator", "10 / 0"),
         ("calculator", "__import__('os').system('ls')"), ("calculator", "9 ** 9 ** 9")]
for name, arg in tests:
    print(f"{name}({arg!r}) ->", str(at.calculator.invoke(arg))[:70])

calculator('48271 * 9371') -> 452347541
calculator('sqrt(2)') -> 1.4142135623730951
calculator('10 / 0') -> Error: ZeroDivisionError: division by zero
calculator("__import__('os').system('ls')") -> Error: ValueError: not allowed: Call(func=Attribute(value=Call(func=Na
calculator('9 ** 9 ** 9') -> Error: ValueError: exponent too large


In [4]:
for code in ["print(sum(range(101)))", "print(1/0)", "while True: pass", "x = 5"]:
    print(repr(code), "->", at.run_python.invoke(code)[:80].replace("\n", " | "))

'print(sum(range(101)))' -> 5050
'print(1/0)' -> Traceback (most recent call last): |   File "<string>", line 1, 


'while True: pass' -> Error: the code did not finish within 5 seconds.
'x = 5' -> (no output; use print to see results)


**What to notice:** the calculator is exact (48271 * 9371 = 452347541) and returns errors as text: division by zero gives 'Error: ZeroDivisionError', a code-injection attempt such as __import__('os').system('ls') is refused because the parser only allows numbers and a short list of operations, and a huge power such as 9 ** 9 ** 9 is refused so it cannot freeze the machine. The code runner returns a traceback as text when the code fails, kills an infinite loop after 5 seconds, and tells the model to print when the code prints nothing. A separate process is not a real security sandbox (the code can still read files), so for untrusted use you need a container, and a person approving risky calls, which is notebook 03.

## 3. The web search tool

`web_search` calls DuckDuckGo through the `ddgs` package (no API key) and formats the top results as plain text. The model reads that text like any other message. Results are cached by query, so the output below is the same on every run; delete `search_cache.json` to see live results.

In [5]:
print(at.web_search.invoke("Who created the Python programming language"))

- History of Python - Wikipedia: Python 2.0, released October 2000, [8] introduced list comprehensions, a feature borrowed from the functional programming languages SETL and Haskell. Python's syntax for this construct is very similar to Haskell's, apart from Haskell's preference for punctuation characters and Python's preference fo (https://en.wikipedia.org/wiki/History_of_Python)
- Guido van Rossum - Wikipedia: According to a programming language popularity survey [36] it is consistently among the top 10 most mentioned languages in job postings. Furthermore, Python has been among the 10 most popular programming languages every year since 2004 according to the TIOBE Programming Community Index and got the n (https://en.wikipedia.org/wiki/Guido_van_Rossum)
- History of Python - GeeksforGeeks: Python is one of the most popular, high-level, general-purpose programming languages. It was created with a focus on code readability and allows developers to write concepts in fewer lines compared

**What to notice:** the tool returns three results as plain text: title, a snippet and the link. The first snippet is about Python 2.0 and not directly about the creator, so the model has to read across results. This text comes from the open web, so it is untrusted: a web page could contain instructions aimed at the model (prompt injection). Treat tool output as data, never as commands.

## 4. What the model sends back when it wants a tool

`llm.bind_tools(tools)` attaches the tool schemas to every request. When the model decides a tool is needed, it does **not** run it. It returns an `AIMessage` with an empty text and a list of `tool_calls`: the tool name and the arguments, as data. Running the function is our job. We ask a question that needs the calculator and inspect the raw reply.

In [6]:
llm_tools = llm.bind_tools(at.TOOLS)
reply = llm_tools.invoke("What is 48271 multiplied by 9371?")
print("text   :", repr(reply.content))
print("calls  :", reply.tool_calls)
print("tokens :", reply.usage_metadata)

text   : ''
calls  : [{'name': 'calculator', 'args': {'expression': '48271 * 9371'}, 'id': 'function-call-17129236266927333983', 'type': 'tool_call'}]
tokens : {'input_tokens': 284, 'output_tokens': 23, 'total_tokens': 307, 'input_token_details': {}, 'output_token_details': {}}


**What to notice:** the model's reply has empty text and one entry in tool_calls: the tool name (calculator), the arguments (expression '48271 * 9371') and an id. It used 284 input tokens and 23 output tokens. The model did no arithmetic itself, it only wrote the expression, and that is why tool use is reliable for calculations: the multiplication is done by code, not by a prediction.

## 5. The agent loop in a dozen lines

An agent is this loop: call the model; if it asked for tools, run them, append the results as `ToolMessage`s and call the model again; if it answered with text, stop. A `max_steps` limit stops a model that never finishes. We write it by hand first so that nothing is hidden. Each message type is shown as it is added.

In [7]:
tool_by_name = {t.name: t for t in at.TOOLS}

def run_agent(question, max_steps=6, verbose=True):
    messages = [HumanMessage(question)]
    used = []
    for step in range(max_steps):
        ai = llm_tools.invoke(messages)                     # 1. the model thinks
        messages.append(ai)
        if not ai.tool_calls:                               # 2. no tool requested: this is the final answer
            return ai.content, used, messages
        for call in ai.tool_calls:                          # 3. run each requested tool, send the result back
            result = tool_by_name[call["name"]].invoke(call["args"])
            used.append(call["name"])
            if verbose:
                print(f"step {step + 1}: {call['name']}({call['args']}) -> {str(result)[:80]!r}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "(stopped: max_steps reached)", used, messages

In [8]:
answer, used, messages = run_agent("What is 15 percent of 2480, plus 7 squared?")
print("\nFINAL:", answer)
print("message types:", [type(m).__name__ for m in messages])

step 1: calculator({'expression': '0.15 * 2480 + 7 ** 2'}) -> '421.0'



FINAL: 15 percent of 2480, plus 7 squared, is 421.0.
message types: ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']


**What to notice:** one tool round gives four messages: HumanMessage, AIMessage (the tool request), ToolMessage (the result) and a final AIMessage (the answer). The model translated '15 percent' into 0.15 and wrote the expression 0.15 * 2480 + 7 ** 2, so it understood the question; the calculator returned 421.0 (a float, hence the .0), and the model reported it.

## 6. Tool choice and multi-step tasks

Now questions that need different tools, and one that needs two in sequence. We print the tools used and the answer.

In [9]:
for q in ["How many days are there between 2024-03-15 and 2025-11-02?",
          "Which company created the LangGraph library? Search the web.",
          "Find the year Python was first released using a web search, then compute how many years before 2025 that was.",
          "What is the capital of France?"]:
    ans, used, _ = run_agent(q, verbose=False)
    print(f"Q: {q}\n   tools: {used}\n   A: {ans[:140]}\n")

Q: How many days are there between 2024-03-15 and 2025-11-02?
   tools: ['run_python']
   A: There are 597 days between 2024-03-15 and 2025-11-02.



Q: Which company created the LangGraph library? Search the web.
   tools: ['web_search']
   A: The LangGraph library was created by LangChain Inc.



Q: Find the year Python was first released using a web search, then compute how many years before 2025 that was.
   tools: ['web_search', 'run_python']
   A: Python was first released in 1991. 2025 - 1991 = 34. So, Python was first released 34 years before 2025.



Q: What is the capital of France?
   tools: ['web_search']
   A: The capital of France is Paris.



**What to notice:** three of the four questions were routed correctly: the date question went to run_python (597 days), the LangGraph question to web_search (created by LangChain Inc.), and the two-step question used web_search and then run_python (1991, so 34 years). The fourth shows a weakness: 'What is the capital of France?' also triggered a web search, a needless call for a well-known fact. Tool over-use costs time and money, and we did not tell the model not to; the next notebook adds that instruction and measures it.

## 7. Experiment: does the tool description matter?

The model chooses tools only from their names and descriptions. We test this directly. For each of 12 questions we know which tool *should* be used (or none). We run the model once with the **good descriptions** written above, and once with **vague descriptions** ("Does math", "Runs code", "Gets info"), and count how often the first tool call is the right one. No tool is executed; we only look at the choice.

In [10]:
from langchain_core.tools import StructuredTool
TASKS = [("What is 48271 multiplied by 9371?", "calculator"), ("What is the square root of 2 to six decimal places?", "calculator"),
         ("What is 15 percent of 2480 plus 7 squared?", "calculator"),
         ("How many days are there between 2024-03-15 and 2025-11-02?", "run_python"),
         ("What is the sum of the first 50 prime numbers?", "run_python"),
         ("Reverse the string 'agentic' and count its vowels.", "run_python"),
         ("Who created the Python programming language? Search the web.", "web_search"),
         ("Which company developed the LangGraph library?", "web_search"),
         ("What is the latest stable version of Python?", "web_search"),
         ("What is the factorial of 20?", "calculator"),
         ("What is the capital of France?", None), ("Hello! Who are you?", None)]

vague = {"calculator": "Does math.", "run_python": "Runs code.", "web_search": "Gets info."}
vague_tools = [StructuredTool.from_function(func=t.func, name=t.name, description=vague[t.name], args_schema=t.args_schema) for t in at.TOOLS]

def first_choice(model, question):
    calls = model.invoke(question).tool_calls
    return calls[0]["name"] if calls else None

rows = []
for q, want in TASKS:
    rows.append(dict(question=q[:52], expected=want, good=first_choice(llm_tools, q), vague=first_choice(llm.bind_tools(vague_tools), q)))
choice = pd.DataFrame(rows)
choice["good_ok"] = choice.expected.eq(choice.good) | (choice.expected.isna() & choice.good.isna())
choice["vague_ok"] = choice.expected.eq(choice.vague) | (choice.expected.isna() & choice.vague.isna())
print("accuracy with good descriptions :", round(choice.good_ok.mean(), 2))
print("accuracy with vague descriptions:", round(choice.vague_ok.mean(), 2))
choice[["question", "expected", "good", "vague"]]

accuracy with good descriptions : 0.92
accuracy with vague descriptions: 0.75


,question,expected,good,vague
0,What is 48271 multiplied by 9371?,calculator,calculator,calculator
1,What is the square root of 2 to six decimal places?,calculator,calculator,calculator
2,What is 15 percent of 2480 plus 7 squared?,calculator,calculator,calculator
3,How many days are there between 2024-03-15 and 2025-,run_python,run_python,calculator
4,What is the sum of the first 50 prime numbers?,run_python,run_python,web_search
5,Reverse the string 'agentic' and count its vowels.,run_python,run_python,run_python
6,Who created the Python programming language? Search,web_search,web_search,web_search
7,Which company developed the LangGraph library?,web_search,web_search,web_search
8,What is the latest stable version of Python?,web_search,web_search,web_search
9,What is the factorial of 20?,calculator,calculator,calculator


**What to notice:** good descriptions gave 11 correct choices out of 12 (0.92); vague descriptions gave 9 (0.75). With 'Does math.' and 'Runs code.' the model sent the date question to the calculator, which cannot handle dates, and the primes question to web_search, which cannot compute it. Both versions made the same mistake on 'capital of France' (a search when none was needed), so the only difference is the two questions that really needed the descriptions. Twelve questions is a small test, but the direction is what matters: the description is part of the program.

## 8. Recap and exercises

- A tool is a function plus a name, a docstring and an argument schema; the docstring is the model's only guide.
- The model only *requests* a tool call; our loop runs it and sends the result back.
- Tools must return errors as text and must have limits (time, allowed operations).
- A `max_steps` guard is essential; the description quality affects tool choice.

**Try yourself:** (1) add a `get_time` tool and ask what day it is; (2) make the calculator description say "use only for money" and see what changes; (3) lower `max_steps` to 1 and ask the two-tool question from section 6.

Next: **Notebook 02** turns this loop into a LangGraph graph with state, memory and an evaluation.